# Huấn luyện LSTM phân tích cảm xúc (Google Colab)

Notebook này là phần **Học sâu** của dự án. Dữ liệu đầu vào là tập `sentiment_dataset.parquet`
do Airflow trích từ **kho dữ liệu** và lưu ở `curated/ml/<category>/` trên Data Lake MinIO.

**Cách dùng**
1. Trên Portal → tab *Data Lake* → bucket `curated` → mở `ml/<category>/sentiment_dataset.parquet` → **Tải về**.
2. Colab: *Runtime → Change runtime type → GPU*.
3. *Runtime → Run all*, chọn file parquet khi được hỏi.
4. Notebook tải về `metrics.json`. Trên Portal → tab *Data Lake* → form **Tải file lên**:
   bucket `curated`, thư mục `models/lstm`. Tab *Khai phá dữ liệu* sẽ tự so sánh LSTM với Naive Bayes và Logistic Regression.

Tập test được giữ **y hệt** tập test của các mô hình ML (cột `split`), nên kết quả so sánh công bằng.

In [ ]:
import datetime
import json
import os
import time

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_recall_fscore_support)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
SEED = 42
tf.keras.utils.set_random_seed(SEED)
LABELS = ["negative", "neutral", "positive"]
QUICK_TEST = False  # True: chạy thử nhanh trên 5.000 mẫu, 1 epoch

## 1. Nạp dữ liệu từ Curated Zone

In [ ]:
DATA_FILE = "sentiment_dataset.parquet"
if not os.path.exists(DATA_FILE):
    from google.colab import files  # chỉ có trên Colab
    uploaded = files.upload()       # chọn file sentiment_dataset.parquet tải từ Portal
    DATA_FILE = next(iter(uploaded))

df = pd.read_parquet(DATA_FILE)
train_df = df[df["split"] == "train"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)  # cùng tập test với Naive Bayes / LR
if QUICK_TEST:
    train_df = train_df.sample(5000, random_state=SEED).reset_index(drop=True)
print(f"{len(train_df):,} train | {len(test_df):,} test")
train_df["sentiment_label"].value_counts()

## 2. Vector hóa văn bản

Dùng cột `text_light` (đã chuyển chữ thường, bỏ HTML/dấu câu, tách *n't → not* nhưng **giữ stopword
và thứ tự từ**) vì LSTM cần ngữ cảnh. Cột `text_clean` (bỏ stopword + lemmatize) dành cho TF-IDF.

In [ ]:
MAX_TOKENS, SEQ_LEN = 30_000, 150
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS, output_sequence_length=SEQ_LEN, standardize=None)  # đã chuẩn hóa ở ETL
vectorizer.adapt(train_df["text_light"].values)

label_id = {label: i for i, label in enumerate(LABELS)}
X_all = vectorizer(train_df["text_light"].values).numpy()
y_all = train_df["sentiment_label"].map(label_id).values
X_test = vectorizer(test_df["text_light"].values).numpy()
y_test = test_df["sentiment_label"].map(label_id).values

# 10% tập train làm validation (phân tầng) để theo dõi overfitting
X_tr, X_val, y_tr, y_val = train_test_split(X_all, y_all, test_size=0.1, stratify=y_all,
                                            random_state=SEED)
# Dữ liệu lệch về 5 sao -> trọng số lớp "balanced" để không bỏ quên lớp Trung tính
class_weight = dict(enumerate(compute_class_weight("balanced", classes=np.arange(3), y=y_tr)))
print("class_weight:", {LABELS[k]: round(v, 2) for k, v in class_weight.items()})

## 3. Mô hình: Embedding → BiLSTM → Dense

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(SEQ_LEN,), dtype="int64"),
    tf.keras.layers.Embedding(MAX_TOKENS, 128, mask_zero=True),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(LABELS), activation="softmax"),
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
EPOCHS = 1 if QUICK_TEST else 8
t0 = time.time()
hist = model.fit(
    X_tr, y_tr, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=256,
    class_weight=class_weight,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2,
                                                restore_best_weights=True)])
train_seconds = round(time.time() - t0, 1)
print(f"Thời gian huấn luyện: {train_seconds} giây")

## 4. Đánh giá trên tập test và xuất `metrics.json`

In [ ]:
pred = model.predict(X_test, batch_size=1024).argmax(axis=1)
y_true = [LABELS[i] for i in y_test]
y_pred = [LABELS[i] for i in pred]
p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=LABELS, zero_division=0)
pm, rm, fm, _ = precision_recall_fscore_support(y_true, y_pred, labels=LABELS, average="macro",
                                                zero_division=0)
metrics = {
    "model": "lstm", "display_name": "BiLSTM", "family": "Deep Learning",
    "trained_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "labels": LABELS,
    "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
    "precision_macro": round(float(pm), 4), "recall_macro": round(float(rm), 4),
    "f1_macro": round(float(fm), 4),
    "f1_weighted": round(float(f1_score(y_true, y_pred, average="weighted")), 4),
    "per_class": {label: {"precision": round(float(p[i]), 4), "recall": round(float(r[i]), 4),
                          "f1": round(float(f[i]), 4), "support": int(s[i])}
                  for i, label in enumerate(LABELS)},
    "confusion_matrix": confusion_matrix(y_true, y_pred, labels=LABELS).tolist(),
    "n_train": int(len(X_tr)), "n_test": int(len(X_test)),
    "features": f"Embedding 128 chiều, chuỗi {SEQ_LEN} token, từ điển {MAX_TOKENS:,}",
    "train_seconds": train_seconds,
    "history": {k: [round(float(x), 4) for x in v] for k, v in hist.history.items()},
    "label_distribution": test_df["sentiment_label"].value_counts().to_dict(),
    "quick_test": QUICK_TEST,
}
print(json.dumps({k: metrics[k] for k in ("accuracy", "precision_macro", "recall_macro", "f1_macro")},
                 indent=2))
pd.DataFrame(metrics["confusion_matrix"], index=LABELS, columns=LABELS)

In [ ]:
with open("metrics.json", "w", encoding="utf-8") as fp:
    json.dump(metrics, fp, ensure_ascii=False, indent=2)
try:
    from google.colab import files
    files.download("metrics.json")  # tải về máy, rồi upload lên curated/models/lstm/ qua Portal
except ImportError:
    print("Đã lưu metrics.json")